# Running PESTPP-IES

In [ ]:
import os
import shutil
import pyemu
import warnings
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import herebedragons as hbd

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [ ]:
# specify the temporary working folder
t_d = os.path.join('pst_template_ies')
# get the weighted PEST dataset + prior ensemble produced by the prior-MC / truth notebook
org_t_d = os.path.join("master_pmc")
if not os.path.exists(org_t_d):
    raise Exception()
if os.path.exists(t_d):
    shutil.rmtree(t_d)
shutil.copytree(org_t_d,t_d)

In [ ]:
pst_path = os.path.join(t_d, 'pest.pst')
pst = pyemu.Pst(pst_path)

In [ ]:
# check to see if obs&weights notebook has been run
if not pst.nnz_obs > 0:
    raise Exception()

Here we set some handy PESTPP-IES options:

- `ies_num_reals`: the number of realizations in the ensemble.
- `ies_multimodal_alpha`: the fraction of the total ensemble used as the local neighbourhood in the multimodal solution process.

(`ies_bad_phi_sigma`, left commented out below, flags a realization as "bad" when its objective function is much worse than the ensemble average — the cutoff is the ensemble standard deviation scaled by this number.)

In [ ]:
pst.pestpp_options["ies_num_reals"] = 100
# pst.pestpp_options["ies_bad_phi_sigma"] = 1.75
pst.pestpp_options["ies_multimodal_alpha"] = 0.99

## Observation noise

History matching needs two *different* facts about each observation, and PESTPP-IES keeps them deliberately apart:

- **Weights** set the **objective function** — how hard each observation pulls on the parameter update, and how the data types are balanced against each other. This is a strategy call (made in the previous notebook, and refined below): we don't let the many head observations drown out the single, management-critical GDE-pH measurement, and we down-weight data the model represents poorly. Weights need *not* equal 1/σ.
- **Noise** sets the **measurement + model error** — how closely we expect the model to be *able* to match each observation. PESTPP-IES draws an *observation+noise ensemble*: every realisation gets its own perturbed target. This stops the ensemble collapsing onto a single best fit, and it carries that uncertainty through to the spread of our forecasts of interest.

In classic PEST these are one number (weight = 1/σ). Pulling them apart matters for decision support: if the noise is too small the ensemble over-fits, the posterior parameter ensemble collapses, and the GDE-pH forecast comes out artificially tight — a confident answer that hides real uncertainty, which is the one thing a decision-maker must not be handed. Too large, and we under-fit and waste the data.

**Why the noise is _correlated_.** Independent, per-observation noise assumes the only thing wrong with our measurements is random, uncorrelated scatter. The larger and more insidious error in a groundwater model is *structural* (model) error — the unavoidable gap between a simplified model and reality (an unmodelled process, an imperfect boundary, coarse discretisation). That error is **systematic**: it pushes whole groups of observations the same way. If we represent error as small independent noise, IES will fit each observation as tightly as it can, and the only way to do that is to contort the parameters to absorb the model error — *parameter compensation* — which collapses the posterior ensemble and bakes (undetectable) bias into the forecasts. A correlated noise component — here, a single shared offset per realisation — says, in effect, "a coherent share of this misfit is the model's fault, not the parameters' — don't chase it." That guards against over-conditioning and protects the forecasts of interest from that bias.

We set the noise explicitly through the `standard_deviation` column: σ = 0.1 for heads (m) and GDE pH (pH units), and *relative* errors for quantities that span orders of magnitude — 20% of the value for the GDE drain flux and 10% for the well chemistry. We then build the correlated observation+noise ensemble and hand it to PESTPP-IES with `ies_obs_en`; supplying it explicitly also makes sure noise is switched **on** (recent PESTPP-IES defaults to *no* noise if none is configured).

In [ ]:
obs = pst.observation_data
obs["standard_deviation"] = np.nan
nzobs = obs.loc[obs.weight > 0, :].copy()

# heads (m) and GDE pH (pH units): sigma = 0.1 absolute; drain flux and chemistry overwritten below (relative)
obs.loc[nzobs.obsnme, "standard_deviation"] = 0.1

# GDE drain flux: sigma = 20% of |value|  (value is negative -> use abs)
dobs = nzobs.loc[nzobs.usecol == "drn-gde", :]
assert dobs.shape[0] == 1
obs.loc[dobs.obsnme, "standard_deviation"] = np.abs(dobs.obsval.values) * 0.2

# well chemistry: sigma = 10% of |value| (relative -- molalities span orders of magnitude)
cobs = nzobs.loc[nzobs.obsnme.str.contains("chemwell"), :]
obs.loc[cobs.obsnme, "standard_deviation"] = np.abs(cobs.obsval.values) * 0.1

# build the correlated observation+noise ensemble (see the discussion above for why
# the noise is correlated, not independent): one common standard-normal factor per
# realization, scaled by each observation's standard deviation
nzobs = obs.loc[obs.weight > 0, :].copy()
np.random.seed(pyemu.en.SEED)
draws = np.random.normal(0.0, 1.0, pst.pestpp_options["ies_num_reals"])
vals = nzobs.obsval.values
stdevs = nzobs.standard_deviation.values
noisyobs = [vals + d * stdevs for d in draws]
df = pd.DataFrame(noisyobs,
                  index=np.arange(pst.pestpp_options["ies_num_reals"]),
                  columns=nzobs.obsnme)
df.to_csv(os.path.join(t_d, "corrnoise.csv"))
pst.pestpp_options["ies_obs_en"] = "corrnoise.csv"

### What the noise looks like

Before running, it is worth *seeing* the noise we just specified. The left panel shows a few representative observations with their noise ensemble, normalised to the measured value (red line at 1.0): the well-determined heads and GDE pH carry a small **absolute** spread (σ = 0.1 in their own units — negligible for a ~90 m head, about a percent for pH ≈ 6.5), while the GDE drain flux and the well chemistry carry a **relative** spread (±20% and ±10% of the value). The right panel makes the two error models explicit — realised $|\text{noise}|$ against measurement magnitude: an **absolute** σ is a flat floor (the same 0.1 whatever the value), a **relative** σ grows in proportion to the measurement. Relative errors suit quantities that span orders of magnitude (fluxes, concentrations), where one fixed absolute σ would be meaningless.

In [ ]:
# Illustrate the noise model just built (df = the observation+noise ensemble)
nz = obs.loc[obs.weight > 0].copy()
def sig_type(row):
    return "relative σ" if (("chemwell" in row.obsnme) or (row.usecol == "drn-gde")) else "absolute σ"
nz["styp"] = nz.apply(sig_type, axis=1)
col = {"absolute σ": "tab:blue", "relative σ": "tab:orange"}

fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)

# Panel A: a few representative obs, noise ensemble normalised to the measured value
def pick(mask):
    s = nz.loc[mask, "obsnme"]
    return s.iloc[0] if len(s) else None
reps = [("head",           pick(nz.obsnme.str.contains("hdslay")), "absolute σ"),
        ("GDE pH",         pick(nz.oname.eq("gde-ph")),            "absolute σ"),
        ("GDE drain flux", pick(nz.usecol.eq("drn-gde")),         "relative σ"),
        ("well chemistry", pick(nz.obsnme.str.contains("chemwell")), "relative σ")]
reps = [(lab, on, st) for lab, on, st in reps if on is not None]
for x, (lab, on, st) in enumerate(reps):
    meas = float(obs.loc[on, "obsval"])
    norm = df.loc[:, on].astype(float).values / meas
    axA.scatter(x + np.random.uniform(-0.09, 0.09, norm.size), norm, s=16, c=col[st], alpha=0.5)
    axA.plot([x - 0.22, x + 0.22], [1, 1], "r-", lw=2.5)
axA.axhline(1.0, color="r", lw=0.6, alpha=0.4)
axA.set_xticks(range(len(reps)))
axA.set_xticklabels([f"{l}\n({s})" for l, _, s in reps], fontsize=8)
axA.set_ylabel("noisy value / measured value")
axA.set_title("Measured value (red) + noise ensemble, normalised")

# Panel B: realised |noise| vs |measured| -- absolute = flat floor, relative = proportional
meas = obs.loc[nz.obsnme, "obsval"].astype(float).values
amag = np.clip(np.abs(meas), 1e-9, None)
adev = np.abs(df.loc[:, nz.obsnme].astype(float).values - meas)
for st in ["absolute σ", "relative σ"]:
    m = nz["styp"].values == st
    axB.scatter(np.repeat(amag[m], adev.shape[0]), np.clip(adev[:, m].T.ravel(), 1e-4, None),
                s=8, c=col[st], alpha=0.25, label=st)
xs = np.logspace(np.log10(amag.min() * 0.7), np.log10(amag.max() * 1.3), 50)
axB.plot(xs, 0.1 * np.ones_like(xs), "--", c="tab:blue",   lw=1.2, label="σ = 0.1 (absolute)")
axB.plot(xs, 0.10 * xs,              "--", c="tab:orange", lw=1.2, label="σ = 10% (chemistry)")
axB.plot(xs, 0.20 * xs,              ":",  c="tab:orange", lw=1.2, label="σ = 20% (drain flux)")
axB.set_xscale("log"); axB.set_yscale("log")
axB.set_xlabel("|measured value|"); axB.set_ylabel("|noise| = |noisy - measured|")
axB.set_title("Realised noise vs magnitude")
axB.legend(fontsize=7)
plt.show()

Let's run PESTPP-IES with `noptmax = -2`. This computes the mean of the initial parameter ensemble, evaluates it (one model run), and records the results.

In [ ]:
pst.control_data.noptmax = -2 
pst.write(os.path.join(t_d, 'pest.pst'),version=2)

pyemu.os_utils.run("pestpp-ies pest.pst",cwd=t_d)

## More weighting adjustments

Weights are our objective-function dial — independent of the noise we just set — so we can keep tuning them. The `ies_phi_factor_file` option turns on internal weight adjustment. It points to a file with two columns and no header (comma- or space-delimited):

- Column 1: a tag matched against observation-group names by substring (lets you combine groups into one "weighting group").
- Column 2: a positive number (phi factor) -- the share of the current measurement phi that group should take.

Here we use four weighting groups -- heads (`hds`), the GDE drain flux (`drn`), the GDE pH (`gde-ph`, the management quantity), and the well chemistry (`chemwell`) -- and concentrate the measurement objective function on the heads and drain flux (see the factors below).

In [ ]:
with open(os.path.join(t_d, "phi.csv"), "w") as f:
    f.write("hds,0.45\n")   # heads
    f.write("drn,0.45\n")   # GDE drain flux
    f.write("gde-ph,0.05\n")    # GDE pH (management quantity)
    f.write("chemwell,0.05\n") #  Wel chem
pst.pestpp_options["ies_phi_factor_file"] = "phi.csv"

In [ ]:
# pst.control_data.noptmax = -2
# pst.write(os.path.join(t_d, 'pest.pst'),version=2)

# pyemu.os_utils.run("pestpp-ies pest.pst",cwd=t_d)

## For the Win!

In [ ]:
pst.control_data.noptmax = 1
pst.write(os.path.join(t_d, 'pest.pst'),version=2)

In [ ]:
num_workers = 20
m_d = "master_ies"

In [ ]:
pyemu.os_utils.start_workers(t_d, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            'pest.pst', # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory
                            )

## Did the misfit come down? The $\Phi$ progression

The first thing to check after any history match is the objective function. PESTPP-IES records the measurement $\Phi$ of every realisation at every iteration (in `pest.phi.actual.csv`, computed against each realisation's *own* noisy targets). The grey lines are the realisations; blue is the ensemble mean.

We want $\Phi$ to fall from its high prior values and then **level off**: once the ensemble matches the data to within the noise, driving $\Phi$ lower would mean fitting the noise — and the model error inside it. A run that keeps pushing $\Phi$ toward zero is over-fitting; one that plateaus high has either uninformative data or a model that cannot reproduce the observations.

In [ ]:
# Objective function (Phi) through the iterations, from pest.phi.actual.csv
pst = pyemu.Pst(os.path.join(m_d, "pest.pst"))
phi = pst.ies.phiactual.copy()
stat = ["iteration", "total_runs", "mean", "standard_deviation", "min", "max", "median"]
reals = [c for c in phi.columns if c not in stat]
it = phi["iteration"].values

fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
for rc in reals:
    ax.plot(it, phi[rc].values, color="0.6", lw=0.4, alpha=0.4)
ax.plot(it, phi["mean"].values, "b-o", lw=2, label="ensemble mean")
ax.plot([], [], color="0.6", lw=1, label="realisations")
vals = phi[reals].values
pos = vals[np.isfinite(vals)]
if pos.size and pos.min() > 0:
    ax.set_yscale("log")
ax.set_xticks(it)
ax.set_xlabel("iteration")
ax.set_ylabel(r"measurement $\Phi$")
ax.set_title("Objective function through history matching")
ax.legend()
plt.show()

## And .. We just did uncertainty

In [ ]:
pst = pyemu.Pst(os.path.join(m_d, "pest.pst"))
obs = pst.observation_data
oe_pr = pst.ies.obsen0
oe_pt = pst.ies.get("obsen", pst.ies.phiactual.iteration.max())
forecasts = [f.strip() for f in pst.pestpp_options["forecasts"].split(",")]

fig, axs = plt.subplots(len(forecasts), 1, figsize=(6, 2.3 * len(forecasts)),
                        constrained_layout=True)
for f, ax in zip(forecasts, np.atleast_1d(axs)):
    oe_pr.loc[:, f].plot(kind="hist", fc="0.5", alpha=0.5, density=True, ax=ax, label="prior")
    oe_pt.loc[:, f].plot(kind="hist", fc="b", alpha=0.5, density=True, ax=ax, label="posterior")
    ylim = ax.get_ylim()
    v = obs.loc[f, "obsval"]
    ax.plot([v, v], ylim, "r-", lw=2, label="truth")
    print(v)
    ax.set_title(" ".join(f.split("usecol:")[-1].split("_")), fontsize=9)
    ax.set_yticks([]); ax.set_ylabel("")
np.atleast_1d(axs)[0].legend()

## Parameter field recovery: prior vs posterior vs truth

For each spatially varying field -- hydraulic conductivity, porosity, and calcite -- compare
the prior, the posterior (after assimilation), and the truth. Tighter posterior-to-truth
agreement means the data informed that field.

In [ ]:
tpst = pyemu.Pst(os.path.join(m_d, "truth.pst"))

In [ ]:
# helper functions to map a field observation set (hk, poro, calcite) to a 2-D array
# and compare prior vs posterior vs truth
def field_obs(oname):
    f = obs.loc[obs.oname == oname, :].copy()
    f["i"] = f.i.astype(int)
    f["j"] = f.j.astype(int)
    return f

def field_to_arr(fobs, series):
    arr = np.full((fobs.i.max() + 1, fobs.j.max() + 1), np.nan)
    arr[fobs.i, fobs.j] = series.loc[fobs.obsnme].values
    return arr

def plot_field(oname, real="base", log=False, title=None):
    fobs = field_obs(oname)
    pr = field_to_arr(fobs, oe_pr.loc[real, :])
    pt = field_to_arr(fobs, oe_pt.loc[real, :])
    tr = field_to_arr(fobs, tpst.res.loc[:, "modelled"])
    if log:
        pr, pt, tr = np.log10(pr), np.log10(pt), np.log10(tr)
    vmin = np.nanmin([pr, pt, tr])
    vmax = np.nanmax([pr, pt, tr])
    lab = title or oname
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    for ax, arr, sub in zip(axes, [pr, pt, tr], ["Prior", "Posterior", "Truth"]):
        cb = ax.imshow(arr, vmin=vmin, vmax=vmax)
        ttl = f"{sub} {lab}" if sub == "Truth" else f"{sub} {lab} (real {real})"
        ax.set_title(ttl)
        plt.colorbar(cb, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

In [ ]:
# parameter field recovery for the base realization
plot_field("hk", real="base", log=True, title="H$_k$ (m/d)")
plot_field("poro", real="base", title="Porosity")
plot_field("calcite", real="base", title="Calcite (mol)")

In [ ]:
# and for another realization
real = oe_pt.index[1]
plot_field("hk", real=real, log=True, title="H$_k$ (m/d)")
plot_field("poro", real=real, title="Porosity")
plot_field("calcite", real=real, title="Calcite (mol)")

## Where did the data inform the parameters?

The realisation maps above are vivid but hard to read in aggregate. Two ensemble statistics summarise what history matching *did* to each spatially varying field:

- **Mean shift** ($\overline{\theta}_{post}-\overline{\theta}_{prior}$) — where the data pulled the field away from the prior expectation (computed in $\log_{10}$ space for the log-distributed $K$ and calcite fields).
- **Relative variance reduction** ($1-\sigma^2_{post}/\sigma^2_{prior}$) — where the data *constrained* the field. Bright ($\rightarrow 1$) means the observations collapsed the spread (well informed); dark ($\rightarrow 0$) means the posterior is as uncertain as the prior (the data say nothing there). This is the spatial fingerprint of data worth.

Read them together: a strong variance reduction with little mean shift means the prior was right and the data confirmed it; a large shift means the prior was off and the data corrected it. Cells that stay dark are where the forecasts of interest must still carry the full prior uncertainty — exactly where a decision-maker should be told the model is guessing.

In [ ]:
# What did history matching do to each spatially varying field?
# (uses field_obs / field_to_arr defined above)
field_cfg = [("hk", "H$_k$", True), ("poro", "Porosity", False), ("calcite", "Calcite", True)]

fig, axes = plt.subplots(len(field_cfg), 2, figsize=(8.5, 4 * len(field_cfg)), constrained_layout=True)
for k, (oname, label, log) in enumerate(field_cfg):
    ax_m, ax_v = axes[k, 0], axes[k, 1]
    fobs = field_obs(oname)
    pr = oe_pr.loc[:, fobs.obsnme].astype(float)
    pt = oe_pt.loc[:, fobs.obsnme].astype(float)
    if log:
        pr, pt = np.log10(pr), np.log10(pt)

    dmean = field_to_arr(fobs, pt.mean() - pr.mean())             # posterior - prior mean
    rvr   = field_to_arr(fobs, 1.0 - pt.var() / pr.var().replace(0, np.nan))  # 1 - var_post/var_prior

    vlim = np.nanmax(np.abs(dmean)) or 1e-9
    cb = ax_m.imshow(dmean, cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    plt.colorbar(cb, ax=ax_m, fraction=0.046, pad=0.04)
    ax_m.set_title(f"{label}: posterior - prior mean" + (r" ($\log_{10}$)" if log else ""))

    cb = ax_v.imshow(np.clip(rvr, 0, 1), cmap="viridis", vmin=0, vmax=1)
    plt.colorbar(cb, ax=ax_v, fraction=0.046, pad=0.04, label=r"$1-\sigma^2_{post}/\sigma^2_{prior}$")
    ax_v.set_title(f"{label}: relative variance reduction")
    for a in (ax_m, ax_v):
        ax_m.set_xticks([]); ax_m.set_yticks([]); ax_v.set_xticks([]); ax_v.set_yticks([])
plt.show()

In [ ]:
fig,ax = plt.subplots(1,1)
noise = pst.ies.noise
hobs = obs.loc[(obs.obsnme.str.contains("hds")) & (obs.weight>0),:]
hobs.sort_values(by="obsval",inplace=True)
hvals = hobs.obsval.values
for real in oe_pt.index:
    vals = noise.loc[real,hobs.obsnme].values
    ax.plot(hvals,vals,"r-",marker=".",alpha=0.5)
    vals = oe_pr.loc[real,hobs.obsnme].values
    ax.plot(hvals, 
            vals,
            "0.5",
            # marker=".",
            alpha=0.5)
    vals = oe_pt.loc[real,hobs.obsnme].values
    ax.plot(hvals, 
            vals,
            "b",
            # marker=".",
            alpha=0.5,
            )

xlim = ax.get_xlim()
ylim = ax.get_ylim()
mn = min(xlim[0],ylim[0])
mx = max(xlim[1],ylim[1])
ax.plot([mn,mx], [mn,mx], "k--", lw=3)
ax.set_xlim(mn,mx)
ax.set_ylim(mn,mx)
ax.set_xlabel("Observed")
ax.set_ylabel("Simulated")
ax.set_title("Observed vs Simulated 1to1")

plt.tight_layout()

    

## Goodness of fit, by target type

The 1:1 plot above is heads only — but a tight fit on heads says nothing about whether the GDE pH or the chemistry were matched. History matching conditioned on **four** target types, so we check each. Below is observed (x) versus simulated (y) for every conditioning group: grey is the prior ensemble, blue the posterior, and the red marker is the observed value with its measurement+model noise. A good fit collapses the blue cloud onto the 1:1 line and down to — but not through — the noise. The single-valued groups (drain flux, GDE pH) appear as a vertical column; watch the blue spread tighten around the observed value relative to the grey prior.

In [ ]:
# Goodness of fit for every conditioning target type (observed vs simulated)
noise = pst.ies.noise
def group_of(row):
    if "chemwell" in row.obsnme: return "well chemistry"
    if row.oname == "gde-ph":    return "GDE pH"
    if row.usecol == "drn-gde":  return "GDE drain flux"
    return "heads"
nz = obs.loc[obs.weight > 0].copy()
nz["grp"] = nz.apply(group_of, axis=1)
groups = [("heads", False), ("GDE drain flux", False), ("GDE pH", False), ("well chemistry", True)]

fig, axes = plt.subplots(1, len(groups), figsize=(4 * len(groups), 4), constrained_layout=True)
for ax, (g, log) in zip(axes, groups):
    onames = nz.loc[nz.grp == g, "obsnme"].tolist()
    if not onames:
        ax.set_visible(False); continue
    ov   = obs.loc[onames, "obsval"].astype(float).values
    nstd = noise.loc[:, onames].std().values
    for oe, c in [(oe_pr, "0.6"), (oe_pt, "b")]:
        sim = oe.loc[:, onames].astype(float).values            # (nreal, nobs)
        ax.scatter(np.tile(ov, sim.shape[0]), sim.ravel(), s=7, c=c, alpha=0.2, edgecolors="none")
    ax.errorbar(ov, ov, yerr=nstd, fmt="r.", ms=6, lw=1, capsize=2, zorder=5)   # observed +/- noise
    allv = np.concatenate([oe_pr.loc[:, onames].values.ravel(),
                           oe_pt.loc[:, onames].values.ravel(), ov]).astype(float)
    allv = allv[np.isfinite(allv)]
    if log:
        allv = allv[allv > 0]
    lo, hi = allv.min(), allv.max()
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    if log:
        ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(g); ax.set_xlabel("observed"); ax.set_ylabel("simulated")
axes[0].scatter([], [], c="0.6", label="prior")
axes[0].scatter([], [], c="b", label="posterior")
axes[0].plot([], [], "r.", label="observed ± noise")
axes[0].legend(fontsize=8)
plt.show()

## Fits to the conditioning targets, by group

History matching does not minimise a single misfit — it balances **four observation groups**, each contributing a share of the objective function: the heads at the well cells, the GDE drain flux, the GDE pH, and the well chemistry. A useful check is how much each group's misfit fell from the prior to the posterior ensemble.

The bars below are the mean measurement objective function ($\Phi$) per group, prior (grey) vs posterior (blue). A group that drops a lot was informative and is now well matched; a group that barely moves was either already satisfied or is being held back by the others — or by the **noise floor**, since we deliberately do *not* fit below the noise (that would be fitting model error, see above). Because $\Phi$ is weighted, the groups are directly comparable despite their different units.

In [ ]:
# Measurement objective function (phi) per conditioning group: prior vs posterior
def group_of(row):
    if "chemwell" in row.obsnme: return "well chemistry"
    if row.oname == "gde-ph":    return "GDE pH"
    if row.usecol == "drn-gde":  return "GDE drain flux"
    return "heads"

nz = obs.loc[obs.weight > 0].copy()
nz["grp"] = nz.apply(group_of, axis=1)

def group_phi(oe, onames):
    w  = obs.loc[onames, "weight"].astype(float).values
    ov = obs.loc[onames, "obsval"].astype(float).values
    r  = oe.loc[:, onames].astype(float).values - ov      # (nreal, nobs) residuals
    return ((r * w) ** 2).sum(axis=1).mean()              # mean weighted phi across realisations

labels, phi_pr, phi_pt = [], [], []
for g in ["heads", "GDE drain flux", "GDE pH", "well chemistry"]:
    onames = nz.loc[nz.grp == g, "obsnme"].tolist()
    if not onames:
        continue
    labels.append(g)
    phi_pr.append(group_phi(oe_pr, onames))
    phi_pt.append(group_phi(oe_pt, onames))

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.bar(x - 0.2, phi_pr, 0.4, color="0.6", label="prior")
ax.bar(x + 0.2, phi_pt, 0.4, color="b",   label="posterior")
if min(phi_pr + phi_pt) > 0:
    ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel(r"mean measurement $\Phi$ (weighted)")
ax.set_title("Misfit by observation group: prior vs posterior")
ax.legend()
plt.show()

## Management quantities through time: prior vs posterior

In [ ]:
phobs = obs.loc[obs.oname == "gde-ph", :].copy()                          # min pH at GDE (1/time)
hobs  = obs.loc[obs.oname == "hdspit", :].copy()                          # 36 pit cells / time
dobs  = obs.loc[(obs.oname == "drn") & (obs.usecol == "drn-gde"), :].copy()  # GDE drain flux (1/time)
for q in (phobs, hobs, dobs):
    q["time"] = q.time.astype(float)

begin_mine, end_mine = 0.0, 10    # active dewatering window (years)

def series(oe, q, agg):
    ts, cols = sorted(q.time.unique()), []
    for t in ts:
        nm  = q.loc[q.time == t, "obsnme"].tolist()
        sub = oe.loc[:, nm]
        cols.append(sub.max(axis=1).values if agg == "max" else sub.iloc[:, 0].values)
    return np.array(ts) / 365.0, np.column_stack(cols)   # yrs, reals x times

def truth(q, agg):
    ts = sorted(q.time.unique())
    return [q.loc[q.time == t, "obsval"].astype(float).max() if agg == "max"
            else float(q.loc[q.time == t, "obsval"].iloc[0]) for t in ts]

panels = [
    ("min pH at GDE",       "pH",                    phobs, "first", 6.0),
    ("max head across pit", "head (m AHD)",          hobs,  "max",   80.0),
    ("GDE drain flux",      "drn-gde flux (m³/day)", dobs,  "first", None),
]

fig, axes = plt.subplots(3, 1, figsize=(6.3, 8), sharex=True, constrained_layout=True)
for ax, (title, ylab, q, agg, crit) in zip(axes, panels):
    yrs, pr = series(oe_pr, q, agg)
    _,   pt = series(oe_pt, q, agg)
    [ax.plot(yrs, pr[i], "0.5", lw=0.5, alpha=0.25) for i in range(pr.shape[0])]
    [ax.plot(yrs, pt[i], "b",   lw=0.5, alpha=0.35) for i in range(pt.shape[0])]
    # ax.plot(yrs, pr[1], "green", lw=0.5, alpha=1)
    ax.plot(yrs, truth(q, agg), "r-", marker="o", lw=2, label="truth")
    ylim = ax.get_ylim()
    ax.plot([begin_mine, begin_mine], ylim, "k--", lw=1, label="mining window")
    ax.plot([end_mine,   end_mine],   ylim, "k--", lw=1)
    if crit is not None:
        ax.axhline(crit, color="crimson", ls=":", lw=1.5, label=f"criterion ({crit:g})")
    ax.plot([], [], "0.5", lw=2, label="prior")        # legend proxies for the spaghetti
    ax.plot([], [], "b",   lw=2, label="posterior")
    ax.set_ylim(ylim)
    ax.set_ylabel(ylab)
    ax.set_title(title, fontsize=10, loc="left")
    ax.legend(fontsize=7, ncol=2, loc="best")
axes[-1].set_xlabel("time (years)")
plt.show()